# Robot Reel quickstart

[Robot Reel](https://github.com/noteflowai/robot-reel) is a physical-AI replay lab: MuJoCo and Newton physics, SmolVLA rollouts, Blender films, an MCP agent director and OpenUSD scenes. This notebook uses only the Python standard library and the recordings shipped in the repository: it checks the included VLA episode, then turns the recorded braking comparison into a checked director storyboard. No GPU, credentials or simulator install are needed, so it runs on a free Colab CPU runtime.

In [ ]:
!git clone --depth 1 https://github.com/noteflowai/robot-reel.git
%cd robot-reel

## Check the recorded VLA episode

`vla docs/vla` reads the recorded SmolVLA rollout (`trace.json`, `manifest.json`) and reports what it contains. Expected output:

```json
{
  "actions": 76,
  "frames": 77,
  "fps": 20,
  "outcome": "success",
  "inference_calls": 8
}
```

In [ ]:
!python3 -m robot_reel.cli vla docs/vla

## Direct the braking comparison

`examples/contact-storyboard.json` splits the 180 recorded source frames into four shots (overview, tracking, half-speed impact, top view). The exporter validates the storyboard against the source, writes the bundle to `artifacts/director` and verifies the hashes. Expected output:

```json
{
  "frames": 210,
  "source_frames": 180,
  "fps": 30
}
```

The 30 extra frames come from the half-speed shot repeating its 30 samples; no motion is generated or dropped.

In [ ]:
!python3 -m robot_reel.cli direct docs/compare/braking --plan examples/contact-storyboard.json --output artifacts/director

## Inspect the exported bundle

The bundle contains `storyboard.json` (the validated plan), `film.json` (the frame-by-frame mapping), `director-manifest.json` (SHA-256 hashes), the Blender build scripts and a `source/` copy of the recording.

In [ ]:
import json, os
from pathlib import Path

bundle = Path('artifacts/director')
for path in sorted(bundle.rglob('*')):
    if path.is_file():
        print(f'{path.stat().st_size:>8}  {path.relative_to(bundle)}')

print()
print(json.dumps(json.loads((bundle / 'storyboard.json').read_text()), indent=2, ensure_ascii=False))

film = json.loads((bundle / 'film.json').read_text())
print()
print({key: value for key, value in film.items() if key != 'frames'})
print('first mapped frames:', film['frames'][:3])

## Next steps

- Published demos and replays: https://noteflowai.github.io/robot-reel/
- Build and render the Blender film from this bundle, or connect an MCP agent as director: [docs/director.md](https://github.com/noteflowai/robot-reel/blob/main/docs/director.md)
- Other example inputs: [examples/README.md](https://github.com/noteflowai/robot-reel/blob/main/examples/README.md)